# Lecture 03: How does a neural network learn?

Run the cells from top to bottom. This notebook uses CPU computation and requires `torch` and `torchvision` in the notebook's Python environment.

If they are missing, run `%pip install torch torchvision` in a new code cell, then restart the kernel.

The MNIST section downloads images into `data/` on its first run. Training takes longer than the small examples. The final two code cells are an optional autoencoder exercise.

## Learning from data

- **Supervised learning:** learn from inputs and target labels.
- **Unsupervised learning:** find structure without target labels.
- **Reinforcement learning:** learn actions from rewards through interaction.
- **Semi-supervised learning:** combine labeled and unlabeled examples.
- **Self-supervised learning:** create prediction targets from the data itself.

Classification predicts a category. Regression predicts a number.

For supervised learning, we need **data**, a **model**, a **loss**, and a **learning algorithm**.

## Dataset or DataLoader?

A Dataset gives individual examples. A DataLoader groups them into batches.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

x = torch.randn(100, 2)  # 100 examples, 2 features
y = (x[:, 0] > 0).long()
dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=10, shuffle=True)

How many batches make one pass through this dataset?

## How do we define a model?

This small network maps two input features to two class scores.

In [ ]:
from torch import nn

model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 2),
)

The output scores are called logits. They are not probabilities.

## Which loss should we use?

Classification: cross-entropy compares class scores with the correct class.

Regression: mean squared error compares predicted and target numbers.

In [ ]:
classification_loss = nn.CrossEntropyLoss()
regression_loss = nn.MSELoss()

The loss must match the prediction task.

## What is cross-entropy?

For one example with true class y:

$$L = -\log p_y, \qquad p_y = \frac{e^{z_y}}{\sum_j e^{z_j}}$$

z is the vector of logits. p_y is the probability of the correct class.

In [ ]:
logits = torch.tensor([[2.0, 0.0]])
target = torch.tensor([0])  # integer class index
loss = nn.CrossEntropyLoss()(logits, target)

Pass raw logits to this loss. It handles log-softmax internally.

## What is mean squared error?

For B examples with one predicted number each:

$$L = \frac{1}{B}\sum_{i=1}^{B}(\hat{y}_i-y_i)^2$$

In [ ]:
prediction = torch.tensor([2.0, 4.0])
target = torch.tensor([1.0, 6.0])
loss = nn.MSELoss()(prediction, target)  # 2.5

PyTorch averages over all output elements by default.

## How does gradient descent work?

The gradient tells us how loss changes with the parameters.

$$\theta_{new} = \theta - \eta\nabla_{\theta}L$$

The learning rate eta controls the step size.

Example: parameter = 2, gradient = 3, learning rate = 0.1.

The updated parameter is 1.7.

## Why use batches?

Full-batch gradient descent uses all training examples for each update.

Stochastic gradient descent uses one randomly sampled example per update.

Mini-batch SGD uses a small group, such as 64 examples.

Batches limit memory use and let us compute several examples together.

One epoch is one pass through the training set.

## How large should a batch be?

Start with batch_size=64 for this MNIST example.

Smaller batches use less memory and give noisier gradient estimates.

Larger batches use more memory and give fewer updates per epoch.

Choose a size that fits memory. Compare learning on validation data.

## How do we predict and measure error?

Continuing the two-feature example:

In [ ]:
xb, yb = next(iter(loader))
logits = model(xb)                 # shape: [10, 2]
predicted_class = logits.argmax(dim=1)
loss = classification_loss(logits, yb)

Use logits to compute the training loss. Use argmax to choose a class.

## How do we compute gradients?

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
optimizer.zero_grad()
loss.backward()

zero_grad clears old gradients. backward computes new gradients.

PyTorch stores each parameter's gradient in its .grad attribute.

backward does not update the parameters.

## How do we update the model?

In [ ]:
optimizer.step()

For plain SGD, step subtracts learning rate times gradient from each parameter.

Then we repeat with another batch.

Why must we clear the old gradients before the next backward call?

## What is a training loop?

In [ ]:
for epoch in range(3):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = classification_loss(logits, yb)
        loss.backward()
        optimizer.step()

Predict, measure loss, compute gradients, update. Repeat.

## MNIST: load the images

This section starts a new MNIST classifier. Run the next four code cells in order.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
transform = transforms.ToTensor()
train = datasets.MNIST("data", train=True,
    download=True, transform=transform)
test = datasets.MNIST("data", train=False,
    download=True, transform=transform)
train_loader = DataLoader(train, batch_size=64, shuffle=True)
test_loader = DataLoader(test, batch_size=256)

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 341kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.15MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.42MB/s]


The first run downloads MNIST into the data folder.

## MNIST: define the model

In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

Each batch has shape [B, 1, 28, 28]. The output has shape [B, 10].

Why does the final layer have 10 outputs?

## MNIST: train the model

In [ ]:
for epoch in range(3):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
    print(epoch + 1, total_loss / len(train))

1 0.4382147904634476
2 0.22604150753815969
3 0.16981377518177032


The printed value is the average training loss for that epoch.

## MNIST: evaluate on new images

In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for xb, yb in test_loader:
        predicted = model(xb).argmax(dim=1)
        correct += (predicted == yb).sum().item()
print(f"Test accuracy: {correct / len(test):.1%}")

Test accuracy: 95.6%


Evaluation uses no parameter updates. no_grad disables gradient tracking.

Keep test data for final evaluation. Use a validation split for tuning.

## Bonus: can we reconstruct an image?

An autoencoder compresses an input into a smaller code, then reconstructs it.

The input image is also the reconstruction target. Digit labels are unnecessary.

In [ ]:
autoencoder = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 32), nn.ReLU(),
    nn.Linear(32, 784), nn.Sigmoid(),
)

The 32 numbers form a bottleneck. What details might it lose?

## Bonus: train an autoencoder

Reuse train_loader from the MNIST example.

In [ ]:
optimizer = torch.optim.SGD(autoencoder.parameters(), lr=0.1)
autoencoder.train()
for xb, _ in train_loader:
    optimizer.zero_grad()
    reconstruction = autoencoder(xb)
    loss = nn.MSELoss()(reconstruction, xb.flatten(1))
    loss.backward()
    optimizer.step()

This is one epoch. Rerun this cell to continue training without resetting the model.

## Is an autoencoder equivalent to PCA?

A linear autoencoder with a smaller bottleneck, centered data, and squared reconstruction error can recover the PCA principal subspace at a global optimum.

Its coordinates need not equal the principal components.

Our ReLU and sigmoid autoencoder is nonlinear. It is not generally equivalent to PCA.

## Check your understanding

1. What enters the model, and what comes out?
2. Why does the classifier's final layer have 10 outputs?
3. Which call computes gradients, and which call updates parameters?
4. Why do we keep the test set separate from training?
5. What changes when an autoencoder learns to reconstruct an image?

## References

- [PyTorch quickstart](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)
- [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)
- [MNIST dataset](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html)
- [Baldi and Hornik (1989): linear networks and PCA](https://doi.org/10.1016/0893-6080(89)90014-2)